In [2]:
# ── Cell 0: install deps + remove the stale torchao, then RESTART ──
!pip install -q gradio transformers peft accelerate bitsandbytes
!pip uninstall -y torchao
print("✅ RESTART now: Runtime > Restart session, then run the demo cell.")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
✅ RESTART now: Runtime > Restart session, then run the demo cell.


In [1]:
# ── Live demo: GPT-2 vs TinyLlama, public URL via share=True ──
!pip install -q gradio transformers peft accelerate bitsandbytes

import torch, gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

HF_USER = "Babblu2821"
GPT2_ADAPTER      = f"{HF_USER}/gpt2-medqa-lora"
TINYLLAMA_ADAPTER = f"{HF_USER}/tinyllama-medqa-qlora"
TINYLLAMA_BASE    = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
HAS_GPU = torch.cuda.is_available()

class DomainChatModel:
    def __init__(self, base_id, adapter_dir, kind, use_4bit=False):
        self.kind = kind
        self.tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        if use_4bit and HAS_GPU:
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_use_double_quant=True,
                                     bnb_4bit_compute_dtype=torch.float16)
            base = AutoModelForCausalLM.from_pretrained(
                base_id, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
        else:
            base = AutoModelForCausalLM.from_pretrained(
                base_id, torch_dtype=torch.float16 if HAS_GPU else torch.float32,
                device_map="auto" if HAS_GPU else None)
        self.model = PeftModel.from_pretrained(base, adapter_dir)
        self.model.eval(); self.model.config.use_cache = True

    def build_prompt(self, q):
        if self.kind == "gpt2":
            return f"### Instruction:\n{q.strip()}\n\n### Response:\n"
        return self.tokenizer.apply_chat_template(
            [{"role": "user", "content": q.strip()}], tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def generate(self, q, max_new_tokens=80):
        inp = self.tokenizer(self.build_prompt(q), return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inp, max_new_tokens=max_new_tokens,
                          do_sample=False,            # greedy: faster + reproducible
                          repetition_penalty=1.15,
                          pad_token_id=self.tokenizer.pad_token_id,
                          eos_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0][inp["input_ids"].shape[1]:],
                                     skip_special_tokens=True).strip()

print("loading models…")
gpt2      = DomainChatModel("gpt2", GPT2_ADAPTER, kind="gpt2", use_4bit=False)
tinyllama = DomainChatModel(TINYLLAMA_BASE, TINYLLAMA_ADAPTER, kind="chat", use_4bit=True)
print("ready ✅")

def compare(question):
    if not question.strip():
        return "Please enter a question.", "Please enter a question."
    return gpt2.generate(question), tinyllama.generate(question)

with gr.Blocks(title="Medical Q&A: GPT-2 vs TinyLlama") as demo:
    gr.Markdown("# 🩺 Medical Q&A — GPT-2 vs TinyLlama (LoRA / QLoRA)")
    gr.Markdown("Same medical dataset, two architectures. "
                "Held-out perplexity: **GPT-2 5.99 → TinyLlama 2.80 (−53%)**")
    q = gr.Textbox(label="Ask a medical question",
                   value="What are the symptoms of Tourette syndrome?")
    btn = gr.Button("Generate", variant="primary")
    with gr.Row():
        a = gr.Textbox(label="GPT-2 (124M · LoRA baseline)", lines=8)
        b = gr.Textbox(label="TinyLlama (1.1B · QLoRA fine-tuned)", lines=8)
    btn.click(compare, inputs=q, outputs=[a, b])
    gr.Markdown("⚠️ *Educational demo of LLM fine-tuning — not medical advice.*")

demo.launch(share=True)   # prints a public https://xxxxx.gradio.live URL (lasts ~72h)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.9 MB/s eta 0:00:00
loading models…


tokenizer_config.json:   0%|          | 0.00/326 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported